In [2]:
import sys
import torch
import numpy as np
import pandas as pd
import torch.nn as nn
from pathlib import Path
import torch.optim as optim
import matplotlib.pyplot as plt
import torchvision.utils as vutils

In [3]:
if torch.cuda.is_available():
    device = torch.device('cuda')
elif torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')

device

device(type='mps')

In [4]:
cwd = Path.cwd()
project_root = cwd.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
    print("Done")

Done


In [5]:
from Scripts.utils import load_mnist_dataset

In [6]:
data_path = project_root / "data"
train_dataloader, test_dataloader = load_mnist_dataset(
    data_path=data_path,
    batch_size=128
)

In [7]:
next(iter(train_dataloader))

[tensor([[[[-1., -1., -1.,  ..., -1., -1., -1.],
           [-1., -1., -1.,  ..., -1., -1., -1.],
           [-1., -1., -1.,  ..., -1., -1., -1.],
           ...,
           [-1., -1., -1.,  ..., -1., -1., -1.],
           [-1., -1., -1.,  ..., -1., -1., -1.],
           [-1., -1., -1.,  ..., -1., -1., -1.]]],
 
 
         [[[-1., -1., -1.,  ..., -1., -1., -1.],
           [-1., -1., -1.,  ..., -1., -1., -1.],
           [-1., -1., -1.,  ..., -1., -1., -1.],
           ...,
           [-1., -1., -1.,  ..., -1., -1., -1.],
           [-1., -1., -1.,  ..., -1., -1., -1.],
           [-1., -1., -1.,  ..., -1., -1., -1.]]],
 
 
         [[[-1., -1., -1.,  ..., -1., -1., -1.],
           [-1., -1., -1.,  ..., -1., -1., -1.],
           [-1., -1., -1.,  ..., -1., -1., -1.],
           ...,
           [-1., -1., -1.,  ..., -1., -1., -1.],
           [-1., -1., -1.,  ..., -1., -1., -1.],
           [-1., -1., -1.,  ..., -1., -1., -1.]]],
 
 
         ...,
 
 
         [[[-1., -1., -1.,  ..., -

In [8]:
class Generator(nn.Module):
    def __init__(self, noise_size: int):
        super().__init__()

        self.tanh   =   nn.Tanh()
        self.relu   =   nn.ReLU()
        self.layer1 =   nn.Linear(in_features=noise_size, out_features=7*7*256)
        self.bn1    =   nn.BatchNorm2d(num_features=256)
        self.layer2 =   nn.ConvTranspose2d(in_channels=256, out_channels=256, kernel_size=3, stride=2, padding=1, output_padding=1)
        self.bn2    =   nn.BatchNorm2d(num_features=256)
        self.layer3 =   nn.ConvTranspose2d(in_channels=256, out_channels=256, kernel_size=3, stride=1, padding=1)
        self.bn3    =   nn.BatchNorm2d(num_features=256)
        self.layer4 =   nn.ConvTranspose2d(in_channels=256, out_channels=256, kernel_size=3, stride=2, padding=1, output_padding=1)
        self.bn4    =   nn.BatchNorm2d(num_features=256)
        self.layer5 =   nn.ConvTranspose2d(in_channels=256, out_channels=1, kernel_size=3, stride=1, padding=1)

    def forward(self, X):                       # (no_of_samples, noise_size)
        X = self.layer1(X)                      # (no_of_samples, 256*7*7)
        X = X.view(X.shape[0], 256, 7, 7)    # (no_of_samples, 256, 7, 7)
        X = self.bn1(X)                         # (no_of_samples, 256, 7, 7)
        X = self.relu(X)                        # (no_of_samples, 256, 7, 7)
        X = self.layer2(X)                      # (no_of_samples, 256, 14, 14)
        X = self.bn2(X)                         # (no_of_samples, 256, 14, 14)
        X = self.relu(X)                        # (no_of_samples, 256, 14, 14)
        X = self.layer3(X)                      # (no_of_samples, 256, 14, 14)
        X = self.bn3(X)                         # (no_of_samples, 256, 14, 14)
        X = self.relu(X)                        # (no_of_samples, 256, 14, 14)
        X = self.layer4(X)                      # (no_of_samples, 256, 28, 28)
        X = self.bn4(X)                         # (no_of_samples, 256, 28, 28)
        X = self.relu(X)                        # (no_of_samples, 256, 28, 28)
        X = self.layer5(X)                      # (no_of_samples, 1, 28, 28)
        X = self.tanh(X)                        # (no_of_samples, 1, 28, 28)

        return X

In [10]:
class Discriminator(nn.Module):
    def __init__(self):
        super().__init__()

        self.lrelu  =   nn.LeakyReLU()
        self.layer1 =   nn.Conv2d(in_channels=1, out_channels=128, kernel_size=5, padding=2, stride=2)
        self.layer2 =   nn.Conv2d(in_channels=128, out_channels=256, kernel_size=5, padding=2, stride=2)
        self.bn1    =   nn.BatchNorm2d(256)
        self.layer3 =   nn.Linear(in_features=256*7*7, out_features=1)

    def forward(self, X):
        X = self.layer1(X)              # (batch_size, 128, 14, 14)
        X = self.lrelu(X)               # (batch_size, 128, 14, 14)
        X = self.layer2(X)              # (batch_size, 256, 7, 7)
        X = self.bn1(X)                 # (batch_size, 256, 7, 7)
        X = self.lrelu(X)               # (batch_size, 256, 7, 7)
        X = X.view(X.shape[0], -1)      # (batch_size, 256*7*7)
        X = self.layer3(X)              # (batch_size, 1)
        
        return X